In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from articulate_anything.agent.actor.mesh_retrieval.partnet_mesh_annotator import *
from dotenv import load_dotenv
from omegaconf import OmegaConf, DictConfig
from articulate_anything.utils.mesh_utils import load_obj, visualize_mesh
from articulate_anything.physics.sapien_simulate import simulate_sapien
from articulate_anything.utils.utils import load_config
from articulate_anything.utils.viz import display_frames

In [ ]:
import os
os.chdir('..')

This notebook goes over the preprocessing steps to prepare the data for mesh retrieval from a text prompt.

`Articulate-Anything` retrieves each part of a 3D object by comparing the textual descriptions between each target mesh part given by a VLM and each mesh part in the PartNet-Mobility library. We will go over the steps to generate the CLIP embedding for each mesh part in the library here. Please see `examples/articulate_text.ipynb` for a step-by-step guide to articulate a particular text prompt once the preprocessing is done.



We will get VLM to auto-label the textual description of the mesh parts in the libary. For that, we have to

1. Generate multiple views of the objects and the parts.
2. Call Gemini to labels the parts.
3. Post-process the meshes


## 1. Part Mesh Rendering

First, we run this script to render multiple views of the objects and each part

In [ ]:
obj_id = "100248"
dataset_dir = "datasets/partnet-mobility-v0/dataset"
partnet_dir = join_path(dataset_dir, obj_id)

In [ ]:
render_partnet_views(partnet_dir);

In [ ]:
pattern = f"{partnet_dir}/part_link_*.jpg"
matching_files = glob.glob(pattern)
images = [Image.open(f) for f in matching_files]
display_frames(images)

## Auto-Labeling the Mesh Parts

In [ ]:
API_KEY = "YOUR-ACTUAL-API-GEMINI-KEY"
## we have our API key stored in a .env file
load_dotenv()
API_KEY = os.environ.get('API_KEY')

In [ ]:
cfg = {
    "out_dir": partnet_dir,
    "api_key": API_KEY,
}
cfg = OmegaConf.create(cfg)

partnet_annotator = PartNetMeshAnnotator(cfg)

In [ ]:
partnet_annotator.cfg.out_dir

In [ ]:
partnet_annotator.generate_prediction(partnet_dir);

In [ ]:
partnet_annotator.load_prediction()

## Post-processing the Meshes

Next, we produce one mesh per part. Note that this step is necessary because each PartNet-Mobility part might consist of many different meshes. Here, we combine all the meshes of a part into a single mesh.

In [ ]:
combine_meshes(partnet_dir)

In [ ]:
vertices, faces = load_obj(f"{partnet_dir}/link_1_combined_mesh.obj")
visualize_mesh(vertices, faces)